## Create environment

In [1]:
%%bash
set -e

cd /content
pwd

curl -fL https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/micromamba.tar.bz2
ls -lh /content/micromamba.tar.bz2

tar -xvjf /content/micromamba.tar.bz2 -C /usr/local/bin --strip-components=1 bin/micromamba

which micromamba
micromamba --version

/content
-rw-r--r-- 1 root root 6.6M Jun  3 13:42 /content/micromamba.tar.bz2
bin/micromamba
/usr/local/bin/micromamba
2.7.0


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  3975    0  3975    0     0   3791      0 --:--:--  0:00:01 --:--:--  3791
100 6722k  100 6722k    0     0  4327k      0  0:00:01  0:00:01 --:--:-- 4327k


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/GWIT2"

# cache persistenti
HF_HOME = f"{PROJECT_ROOT}/.cache/huggingface"
PIP_CACHE_DIR = f"{PROJECT_ROOT}/.cache/pip"

# runtime locale
ENV_PATH = "/content/micromamba/envs/gwit"
LOCAL_ROOT = "/content/gwit_runtime"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(PIP_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_ROOT, exist_ok=True)
os.makedirs(LOCAL_LATENTS, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("ENV_PATH     =", ENV_PATH)
print("HF_HOME      =", HF_HOME)

PROJECT_ROOT = /content/drive/MyDrive/GWIT2
ENV_PATH     = /content/micromamba/envs/gwit
HF_HOME      = /content/drive/MyDrive/GWIT2/.cache/huggingface


In [4]:
import os

os.environ["HF_HOME"] = HF_HOME
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{HF_HOME}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{HF_HOME}/datasets"
os.environ["PIP_CACHE_DIR"] = PIP_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("PIP_CACHE_DIR =", os.environ["PIP_CACHE_DIR"])

HF_HOME = /content/drive/MyDrive/GWIT2/.cache/huggingface
HF_DATASETS_CACHE = /content/drive/MyDrive/GWIT2/.cache/huggingface/datasets
PIP_CACHE_DIR = /content/drive/MyDrive/GWIT2/.cache/pip


In [5]:
acc_dir = f"{HF_HOME}/accelerate"
os.makedirs(acc_dir, exist_ok=True)

config_text = """compute_environment: LOCAL_MACHINE
distributed_type: NO
machine_rank: 0
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
"""

with open(f"{acc_dir}/default_config.yaml", "w") as f:
    f.write(config_text)

print("✔ accelerate default_config.yaml creato in", acc_dir)

✔ accelerate default_config.yaml creato in /content/drive/MyDrive/GWIT2/.cache/huggingface/accelerate


In [6]:
import os

if not os.path.isdir(ENV_PATH):
    print("Creo environment locale...")
    !micromamba create -y -p "{ENV_PATH}" python=3.10 pip
    !micromamba run -p "{ENV_PATH}" pip install --upgrade pip setuptools wheel

    # Stack moderno: lascia che xformers installi torch compatibile
    !micromamba run -p "{ENV_PATH}" pip install xformers

    # Installa torchvision coerente con il torch già installato
    !micromamba run -p "{ENV_PATH}" pip install torchvision --upgrade

    # Pacchetti principali
    !micromamba run -p "{ENV_PATH}" pip install \
      accelerate \
      diffusers \
      "transformers<5" \
      datasets \
      huggingface_hub \
      hf_transfer \
      "safetensors>=0.4.5,<0.7" \
      numpy \
      scipy \
      scikit-learn \
      pillow \
      tqdm \
      tensorboard \
      wandb \
      einops \
      packaging \
      dalle2-pytorch \
      kornia \
      opencv-python \
      matplotlib \
      pandas \
      lpips \
      torchmetrics \
      pytorch-fid \
      clean-fid \
      ftfy \
      regex \
      timm

    # Check finale
    !micromamba run -p "{ENV_PATH}" pip check

else:
    print("Environment locale già presente in questa sessione.")

Creo environment locale...
[+] 0.0s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (4.2 sec)

Resolving Environment                                                                     ✔ Done (0.1 sec)

Transaction

  Prefix: /content/micromamba/envs/gwit

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_9            conda-forge     260kB
  + ca-certificates   2026.5.20 

In [ ]:
DRIVE_LATENTS = f"{PROJECT_ROOT}/data/latents"
LOCAL_LATENTS = f"{LOCAL_ROOT}/latents"

!mkdir -p "{LOCAL_LATENTS}"
!rsync -ah --delete --info=progress2 "{DRIVE_LATENTS}/" "{LOCAL_LATENTS}/"

In [7]:
DRIVE_CLIP_EMBEDS = f"{PROJECT_ROOT}/data/clip_embeds_openclip_zebra"
LOCAL_CLIP_EMBEDS = f"{LOCAL_ROOT}/clip_embeds"

!mkdir -p "{LOCAL_CLIP_EMBEDS}"
!rsync -ah --delete --info=progress2 "{DRIVE_CLIP_EMBEDS}/" "{LOCAL_CLIP_EMBEDS}/"

         20.41G 100%   52.18MB/s    0:06:12 (xfr#24, to-chk=0/32)


## Precompute latents

In [ ]:
%cd "/content/drive/My Drive/GWIT2"

!micromamba run -n gwit python3 precompute_latents.py \
  --model_name Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --batch_size 16 \
  --save_every 25 \
  --output_root data/latents

/content/drive/My Drive/GWIT2
[INFO] Loading full HF pool: train + validation + test
Generating test split: 100% 1987/1987 [00:02<00:00, 763.57 examples/s]
Generating train split: 100% 7959/7959 [00:13<00:00, 598.56 examples/s]
Generating validation split: 100% 1994/1994 [00:07<00:00, 263.73 examples/s]
[INFO] Full pool size: 11940 samples
[INFO] Loading VAE...
config.json: 100% 553/553 [00:00<00:00, 344kB/s]
diffusion_pytorch_model.safetensors: 100% 335M/335M [00:09<00:00, 35.4MB/s]
[INFO] Processing subject 1
Filter: 100% 11940/11940 [01:17<00:00, 154.82 examples/s]
[INFO] Subject 1: 1980 samples in full pool
[INFO] Starting latent computation for subj1 with batch_size=16
subj1:  19% 24/124 [01:31<06:18,  3.78s/it][CHECKPOINT] subj1: saved 400 / 1980
subj1:  40% 49/124 [03:07<04:47,  3.84s/it][CHECKPOINT] subj1: saved 800 / 1980
subj1:  60% 74/124 [04:44<03:11,  3.83s/it][CHECKPOINT] subj1: saved 1200 / 1980
subj1:  80% 99/124 [06:21<01:36,  3.88s/it][CHECKPOINT] subj1: saved 1600 / 

## Precompute OpenCLIP embeddings

Image embeds

In [ ]:
!micromamba run -p "{ENV_PATH}" python precompute_openclip_embeds_zebra.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --image_column image \
  --cache_dir "{HF_HOME}" \
  --batch_size 16 \
  --output_root /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra \
  --arch ViT-bigG-14 \
  --pretrained laion2b_s39b_b160k

[INFO] device = cuda
[INFO] Loading full HF pool for luigi-s/EEG_Image_CVPR_ALL_subj ...
[INFO] Total samples in full HF pool: 11940
[INFO] Loading FrozenOpenCLIPImageEmbedder...
[INFO] Dummy embedder output shape: (2, 256, 1664)
[INFO] Dummy token count T = 256
[INFO] Dummy embed dim   D = 1664
[INFO] subj1: 1980 samples
[INFO] subj1: creating new memmap with shape (1980, 256, 1664)
subj1:   0% 0/124 [00:00<?, ?it/s][INFO] subj1 first batch token shape: (16, 256, 1664) (B=16, T=256, D=1664)
subj1:   7% 9/124 [00:15<03:08,  1.64s/it][INFO] subj1: flushed memmap at 160/1980
subj1:  15% 19/124 [00:31<02:53,  1.65s/it][INFO] subj1: flushed memmap at 320/1980
subj1:  23% 29/124 [00:49<02:42,  1.71s/it][INFO] subj1: flushed memmap at 480/1980
subj1:  31% 39/124 [01:07<02:32,  1.79s/it][INFO] subj1: flushed memmap at 640/1980
subj1:  40% 49/124 [01:26<02:24,  1.92s/it][INFO] subj1: flushed memmap at 800/1980
subj1:  48% 59/124 [01:45<02:00,  1.85s/it][INFO] subj1: flushed memmap at 960/1980


Text embeds

In [ ]:
!micromamba run -p "{ENV_PATH}" python precompute_openclip_text_embeds_zebra.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --subjects 1 2 3 4 5 6 \
  --caption_column caption \
  --cache_dir "{HF_HOME}" \
  --batch_size 64 \
  --output_root /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra \
  --arch ViT-bigG-14 \
  --pretrained laion2b_s39b_b160k

[INFO] device = cuda
[INFO] Loading full HF pool for luigi-s/EEG_Image_CVPR_ALL_subj ...
[INFO] Total samples in full HF pool: 11940
[INFO] Loading FrozenOpenCLIPEmbedder2...
[INFO] Dummy pooled text output shape: (2, 1280)
[INFO] Dummy pooled embed dim D = 1280
[INFO] subj1: 1980 samples
[INFO] subj1: creating new memmap with shape (1980, 1280)
subj1:   0% 0/31 [00:00<?, ?it/s][INFO] subj1 first batch pooled text shape: (64, 1280) (B=64, D=1280)
subj1:  61% 19/31 [00:33<00:23,  1.94s/it][INFO] subj1: flushed memmap at 1280/1980
subj1: 100% 31/31 [00:39<00:00,  1.29s/it]
[INFO] subj1: converting memmap to final .npy ...
[DONE] subj1 -> shape=(1980, 1280)
[DONE] saved: /content/drive/MyDrive/GWIT2/data/clip_embeds_openclip_zebra/luigi-s_EEG_Image_CVPR_ALL_subj/subj1/clip_text_embeds.npy
[INFO] subj2: 1992 samples
[INFO] subj2: creating new memmap with shape (1992, 1280)
subj2:   0% 0/32 [00:00<?, ?it/s][INFO] subj2 first batch pooled text shape: (64, 1280) (B=64, D=1280)
subj2:  59% 19/

## Train

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Stage 1 — EEG backbone + SIFE + reconstruction

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage1 \
  --report_to wandb \
  --tracker_project_name zebra_stage1 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 128 \
  --val_batch_size 128 \
  --gradient_accumulation_steps 1 \
  --validation_steps 500 \
  --checkpointing_steps 2000 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --eeg_backbone_lr 1e-4 \
  --sife_lr 1e-4 \
  --recon_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/data/eegfeat_cvpr.pth \
  --use_sife \
  --use_eeg_reconstruction \
  --train_eeg_only \
  --training_stage stage1 \
  --grl_lambda_sife 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --num_train_epochs 60

2026-04-17 09:56:25,975 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Subject 3 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Subject 4 | mode=train | kept 1795/1994 | cap=None
[EEG DATASET] Subject 5 | mode=train | kept 1792/1991 |

### Stage 2 — SSFE

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage2 \
  --report_to wandb \
  --tracker_project_name zebra_stage2 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 128 \
  --val_batch_size 128 \
  --gradient_accumulation_steps 1 \
  --validation_steps 500 \
  --checkpointing_steps 500 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --ssfe_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/output/zebra_stage1/eeg_backbone.pt \
  --load_sife_path /content/drive/MyDrive/GWIT2/output/zebra_stage1/sife.pt \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_ssfe \
  --train_eeg_only \
  --training_stage stage2 \
  --ssfe_adapter_type zebra_like \
  --grl_lambda_ssfe 1.0 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_text 0.25 \
  --num_train_epochs 120 \
  --resume_from_checkpoint=checkpoint-4000

2026-04-28 18:38:04,993 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj1: shape=(1980, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj1: shape=(1980, 1280)
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP i

### Stage 3 — Prior

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v3 \
  --report_to wandb \
  --tracker_project_name zebra_stage3_v3 \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 8 \
  --val_batch_size 8 \
  --gradient_accumulation_steps 16 \
  --validation_steps 200 \
  --checkpointing_steps 150 \
  --console_log_every 100 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --prior_lr 1e-4 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/output/zebra_stage1/eeg_backbone.pt \
  --load_sife_path /content/drive/MyDrive/GWIT2/output/zebra_stage1/sife.pt \
  --load_ssfe_path /content/drive/MyDrive/GWIT2/output/zebra_stage2/ssfe_projector.pt \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_ssfe \
  --use_prior \
  --train_eeg_only \
  --training_stage stage3 \
  --lambda_prior 1.0 \
  --num_train_epochs 200 \
  --resume_from_checkpoint=checkpoint-13500

2026-05-07 15:06:29,167 - INFO - Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj1: shape=(1980, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj1: shape=(1980, 1280)
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP i

### Generate images

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3 \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_recon_test_subj6 \
  --test_subjects 6 \
  --batch_size 32 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --num_samples_per_image 1 \
  --save_vis \
  --save_pt \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000.ckpt

Fase A

In [ ]:
!micromamba run -p "{ENV_PATH}" python recon_stageA_tokens.py \
  --pretrained_model_name_or_path Manojb/stable-diffusion-2-1-base \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root /content/drive/MyDrive/GWIT2/data \
  --model_dir /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2 \
  --output_dir /content/drive/MyDrive/GWIT2/output/recon_stageA_subj6 \
  --tmp_dir /content/gwit_runtime/recon_stageA_tmp \
  --test_subjects 6 \
  --batch_size 64 \
  --prior_inference_steps 20 \
  --prior_cond_scale 1.0 \
  --save_gts \
  --resume

/content/micromamba/envs/gwit/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [6]: 1991 samples
[EEG DATASET] Subject 6 | mode=test | kept 1991/1991 | cap=None
[EEG DATASET] Final dataset | mode=test | subjects=[6] | total=1991
[EEG Backbone Load]
Source: /content/drive/MyDrive/GWIT2/output/zebra_stage3_v2/eeg_backbone.pt
Missing keys: []
Unexpected keys: []
[RESUME] found 0 completed shard(s)
[RAM] before loop used=3.51 GB | avail=9.16 GB
Stage A: generating prior tokens:   0% 0/32 [00:00<?, ?it/s][RAM] after shard 00000 used=3.80 GB | avail=8.87 GB
Stage A: generating prior 

Fase B

In [ ]:
!MPLBACKEND=Agg micromamba run -p "{ENV_PATH}" python recon_stageB_unclip.py \
  --stageA_dir /content/drive/MyDrive/GWIT2/output/recon_stageA_subj6 \
  --output_dir /content/drive/MyDrive/GWIT2/output/recon_stageB_subj6 \
  --unclip_ckpt /content/drive/MyDrive/GWIT2/data/unclip6_epoch0_step110000_filtered_for_recon.safetensors \
  --unclip_config /content/drive/MyDrive/GWIT2/models/generative_models/configs/unclip6.yaml \
  --num_samples_per_image 1 \
  --decode_batch_size 1 \
  --model_dtype fp32 \
  --resume \
  --resume_minibatch \
  --save_final_manifest \
  --save_vis \
  --log_ram

/content/micromamba/envs/gwit/lib/python3.9/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
[RAM] before prepare_unclip | used=1.71 GB | avail=10.96 GB
[RAM] before OmegaConf.load | used=1.71 GB | avail=10.96 GB
[RAM] before MinimalUnclipEngine init | used=1.72 GB | avail=10.95 GB
No SDP backend available, likely because you are running in pytorch versions < 2.0. In fact, you are using PyTorch 1.13.1+cu117. You might want to consider upgrading.
SpatialTransformer: Found context dims [1664] of depth 1, which does not match the sp

## New Train

In [8]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lorenzosoannini (lorenzosoannini-sapienza-universit-di-roma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Stage 1

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train_eeg_only.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage1_colab_new \
  --report_to wandb \
  --tracker_project_name zebra_stage1_colab_new \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 128 \
  --val_batch_size 128 \
  --gradient_accumulation_steps 1 \
  --validation_steps 250 \
  --checkpointing_steps 1000 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --eeg_backbone_lr 1e-4 \
  --sife_lr 1e-4 \
  --recon_lr 1e-4 \
  --eeg_backbone_ckpt data/eegfeat_cvpr.pth \
  --use_stage1_clip_pretrain \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_eeg_reconstruction \
  --training_stage stage1 \
  --grl_lambda_sife 1.0 \
  --lambda_subject_inv 1.0 \
  --lambda_subject_spec 1.0 \
  --lambda_recon 0.5 \
  --num_train_epochs 30

2026-06-03 14:05:40,908 - INFO - [RANK 0] Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

[EEG DATASET] Loaded full HF pool: 11940 samples
[EEG DATASET] After subject index map [1, 2, 3, 4, 5]: 9949 samples
[EEG DATASET] Subject 1 | mode=train | kept 1782/1980 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj1: shape=(1980, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj1: shape=(1980, 1280)
[EEG DATASET] Subject 2 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj2: shape=(1992, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj2: shape=(1992, 1280)
[EEG DATASET] Subject 3 | mode=train | kept 1793/1992 | cap=None
[EEG DATASET] Loaded CLIP image embeds for subj3: shape=(1992, 256, 1664) | mode=sequence
[EEG DATASET] Loaded CLIP text embeds for subj3: shape=(1992, 1280)
[EEG DATASET] Subject 4 | mode=train | kept 1

### Stage 2 joint

In [ ]:
!micromamba run -p "{ENV_PATH}" accelerate launch train_eeg_only.py \
  --dataset_name luigi-s/EEG_Image_CVPR_ALL_subj \
  --data_root data \
  --output_dir /content/drive/MyDrive/GWIT2/output/zebra_stage2_joint_blurry_colab \
  --report_to wandb \
  --tracker_project_name zebra_stage2_joint_blurry_colab \
  --train_subjects 1 2 3 4 5 \
  --val_subjects 1 2 3 4 5 \
  --test_subjects 6 \
  --val_ratio 0.1 \
  --split_seed 42 \
  --seed 42 \
  --train_batch_size 32 \
  --val_batch_size 32 \
  --gradient_accumulation_steps 4 \
  --validation_steps 200 \
  --checkpointing_steps 500 \
  --console_log_every 200 \
  --mixed_precision fp16 \
  --learning_rate 1e-4 \
  --ssfe_lr 1e-4 \
  --prior_lr 1e-4 \
  --blurry_lr 1e-4 \
  --lr_scheduler cosine \
  --lr_warmup_steps 500 \
  --eeg_backbone_ckpt /content/drive/MyDrive/GWIT2/output/zebra_stage1_colab_new/eeg_backbone.pt \
  --load_sife_path /content/drive/MyDrive/GWIT2/output/zebra_stage1_colab_new/sife.pt \
  --init_general_projector_from_stage1 /content/drive/MyDrive/GWIT2/output/zebra_stage1_colab_new/stage1_clip_projector.pt \
  --use_precomputed_clip_embeds \
  --clip_embeds_dir /content/gwit_runtime/clip_embeds \
  --use_sife \
  --use_ssfe \
  --use_prior \
  --use_blurry_recon \
  --blurry_autoenc_path data/blurry_autoencoder.pt \
  --training_stage stage2_joint \
  --ssfe_adapter_type zebra_like \
  --image_dis_grl_mode full \
  --grl_lambda_ssfe 1.0 \
  --lambda_ssfe 1.0 \
  --lambda_image_cls 1.0 \
  --lambda_image_dis 1.0 \
  --lambda_anchor_cls 0.5 \
  --lambda_anchor_visual 0.5 \
  --lambda_anchor_visual_s 1.0 \
  --lambda_anchor_text 0.25 \
  --lambda_prior 30.0 \
  --lambda_blurry 0.5 \
  --prior_depth 6 \
  --prior_timesteps 100 \
  --prior_cond_drop_prob 0.2 \
  --gradient_checkpointing_ssfe \
  --gradient_checkpointing_prior \
  --num_train_epochs 60